In [1]:
import json
import pandas as pd
import os
import re

# -----------------------------------------------------------
# 1. 파일 경로 설정
# -----------------------------------------------------------
json_file_path = '../SSU_Datathon2025_공학분야_62199.json'

file_paths_if = {
    2021: '2021_인용지수_2년분.xls',
    2022: '2022_인용지수_2년분.xls',
    2023: '2023_인용지수_2년분.xls',
    2024: '2024_인용지수_2년분.xls'
}

# -----------------------------------------------------------
# 2. 사용자 수기 확인 내역 (매핑 테이블) 적용
# -----------------------------------------------------------
manual_mapping = {
    "(사)한국CDE학회": "한국CDE학회",
    "ICT플랫폼학회": "아이씨티플랫폼학회",
    "유공압건설기계학회": "사단법인 유공압건설기계학회",
    "한국로봇학회(논문지)": "한국로봇학회",
    "한국염색가공학회": "한국염색가공학회",
    "한국위험물학회": "한국위험물학회",
    "한국자동차안전학회": "사단법인 한국자동차안전학회",
    "한국전자파학회JEES": "한국전자파학회",
    "한국정보통신학회JICCE": "한국정보통신학회",
    "한국컴퓨터그래픽스학회": "(사)한국컴퓨터그래픽스학회",
    "한국콘텐츠학회(IJOC)": "한국콘텐츠학회",
    "한국환경에너지공학회": "(사)한국환경에너지공학회"
}

# -----------------------------------------------------------
# 3. 유틸리티 함수
# -----------------------------------------------------------
def normalize_name(name):
    """매칭 확률을 높이기 위해 공백과 특수문자를 제거"""
    if pd.isna(name): return ""
    return re.sub(r'[^a-zA-Z0-9가-힣]', '', str(name).upper())

def load_if_database(file_paths):
    """인용지수 DB 구축"""
    db = {}
    print("📂 인용지수 데이터베이스 구축 중...")
    
    for year, path in file_paths.items():
        if not os.path.exists(path):
            db[year] = {}
            continue
            
        try:
            df = pd.read_excel(path, engine='xlrd')
            df.columns = df.columns.str.replace('\n', '').str.strip()
            
            # IF 컬럼 찾기
            if_cols = [c for c in df.columns if '2년' in c and 'IF' in c]
            if not if_cols:
                db[year] = {}
                continue
            if_col = if_cols[0]
            
            # 발행기관 컬럼 찾기
            org_cols = [c for c in df.columns if any(x in c for x in ['발행기관', '발행처', '학회'])]
            target_col = org_cols[0] if org_cols else (df.columns[1] if len(df.columns) > 1 else df.columns[0])

            # 점수 매핑
            mapping = {}
            df[if_col] = pd.to_numeric(df[if_col], errors='coerce').fillna(0)
            
            for idx, row in df.iterrows():
                org_name = row[target_col]
                score = row[if_col]
                norm_name = normalize_name(org_name)
                
                if norm_name:
                    mapping[norm_name] = max(mapping.get(norm_name, 0), score)
            
            db[year] = mapping
            
        except Exception as e:
            print(f"   ❌ {year}년 로드 실패: {e}")
            db[year] = {}
            
    db[2025] = db.get(2024, {})
    return db

def main():
    # 1. 인용지수 DB 로드
    if_db = load_if_database(file_paths_if)
    
    # 2. JSON 논문 데이터 로드
    print(f"📂 논문 데이터 로드 중... ({json_file_path})")
    try:
        with open(json_file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
        if "NODE_LIST" not in data:
            print("오류: NODE_LIST 키가 없습니다.")
            return

        df = pd.DataFrame(data["NODE_LIST"])
        
        # 분석을 위해 임시로 Year 컬럼 생성
        df['temp_Year'] = df['PBSH'].astype(str).str.strip().str[:4]
        
        # 3. IF 점수 매핑 (보정 로직 적용)
        print("📊 보정된 매핑 테이블을 사용하여 IF 점수 계산 중...")
        
        def get_score(row):
            try:
                y = int(row['temp_Year'])
            except:
                y = 0
            
            original_name = row['IPRD_NM']
            
            # [수정됨] 1. 수기 매핑 테이블 확인
            if original_name in manual_mapping:
                search_name = manual_mapping[original_name]
            else:
                search_name = original_name
                
            # 2. 정규화 및 점수 조회
            norm_name = normalize_name(search_name)
            return if_db.get(y, {}).get(norm_name, 0)

        df['IF_Score'] = df.apply(get_score, axis=1)
        
        # 4. 상위 30% 추출
        print("✂️ [연도 x 중분류] 별 상위 30% 필터링 시작...")
        
        filtered_results = []
        groups = df.groupby(['temp_Year', 'NODE_CLSS_02'])
        
        for (year, clss), group in groups:
            total_count = len(group)
            target_count = int(total_count * 0.3)
            
            if target_count == 0:
                continue

            # 정렬: IF 점수 내림차순 -> NODE_ID 오름차순
            sorted_group = group.sort_values(by=['IF_Score', 'NODE_ID'], ascending=[False, True])
            
            # 자르기
            top_30 = sorted_group.head(target_count)
            filtered_results.append(top_30)

        # 5. 저장
        if filtered_results:
            final_df = pd.concat(filtered_results)
            
            # [중요] 원본 데이터 형태 유지를 위해 임시 컬럼(temp_Year) 삭제
            if 'temp_Year' in final_df.columns:
                final_df = final_df.drop(columns=['temp_Year'])
            
            # 검증을 위해 IF_Score는 남겨두는 것을 권장하지만, 원본과 100% 동일한 필드를 원하시면 아래 주석 해제
            # final_df = final_df.drop(columns=['IF_Score'])
            
            output_json = 'top_30_percent_corrected.json'
            
            # JSON 저장
            final_df.to_json(output_json, orient='records', force_ascii=False, indent=4)
            
            print("\n" + "="*50)
            print(f"🎉 저장 완료! (수기 보정 적용됨)")
            print(f"- 원본 데이터: {len(df)}건")
            print(f"- 추출 데이터: {len(final_df)}건 (상위 30%)")
            print(f"- 파일명: {output_json}")
            print("="*50)
            
            # 저장된 데이터의 컬럼 목록 확인
            print("[저장된 필드 목록]")
            print(final_df.columns.tolist())
            
        else:
            print("조건에 맞는 논문이 없습니다.")

    except Exception as e:
        print(f"에러 발생: {e}")

if __name__ == "__main__":
    main()

📂 인용지수 데이터베이스 구축 중...
WARNING *** OLE2 stream 'SSCS': expected size 132288, actual size 512
WARNING *** OLE2 stream 'SSCS': expected size 133824, actual size 512
WARNING *** OLE2 stream 'SSCS': expected size 132288, actual size 512
WARNING *** OLE2 stream 'SSCS': expected size 133312, actual size 512
📂 논문 데이터 로드 중... (../SSU_Datathon2025_공학분야_62199.json)
📊 보정된 매핑 테이블을 사용하여 IF 점수 계산 중...
✂️ [연도 x 중분류] 별 상위 70% 필터링 시작...
에러 발생: name 'top_70' is not defined


In [6]:
# 검증 코드

import json
import pandas as pd
import math

# 1. 파일 경로 설정
original_file = '../SSU_Datathon2025_공학분야_62199.json'   # 원본 파일
extracted_file = 'top_30_percent_corrected.json'           # 방금 만든 추출 파일

def verify_extraction():
    print("🔍 검증 작업을 시작합니다...\n")

    # -------------------------------------------------------
    # 1. 원본 데이터 분석 (목표치 계산)
    # -------------------------------------------------------
    try:
        with open(original_file, 'r', encoding='utf-8') as f:
            data_org = json.load(f)
            df_org = pd.DataFrame(data_org["NODE_LIST"])
            
        # 연도 추출
        df_org['Year'] = df_org['PBSH'].astype(str).str.strip().str[:4]
        
        # 그룹별 전체 개수 카운트
        # reset_index를 해서 데이터프레임으로 만듦
        org_stats = df_org.groupby(['Year', 'NODE_CLSS_02']).size().reset_index(name='Total_Count')
        
        # 목표 개수(Target) 계산: 전체 * 0.3 (소수점 내림)
        org_stats['Target_Count'] = (org_stats['Total_Count'] * 0.3).astype(int)
        
        print(f"✅ 원본 데이터 로드 완료 ({len(df_org)}건)")

    except Exception as e:
        print(f"❌ 원본 파일 로드 실패: {e}")
        return

    # -------------------------------------------------------
    # 2. 추출된 데이터 분석 (실제값 확인)
    # -------------------------------------------------------
    try:
        # JSON 파일 로드
        df_ext = pd.read_json(extracted_file)
        
        # 연도 추출 (만약 저장할 때 temp_Year를 지웠다면 다시 PBSH에서 추출)
        if 'Year' not in df_ext.columns:
             df_ext['Year'] = df_ext['PBSH'].astype(str).str.strip().str[:4]

        # 그룹별 추출 개수 카운트
        ext_stats = df_ext.groupby(['Year', 'NODE_CLSS_02']).size().reset_index(name='Actual_Count')
        
        print(f"✅ 추출 데이터 로드 완료 ({len(df_ext)}건)")

    except ValueError:
        print("❌ 추출된 파일이 비어있거나 형식이 잘못되었습니다.")
        return
    except Exception as e:
        print(f"❌ 추출 파일 로드 실패: {e}")
        return

    # -------------------------------------------------------
    # 3. 비교 검증 (Merge & Compare)
    # -------------------------------------------------------
    print("\n📊 검증 결과 집계 중...")
    
    # 원본 통계(org_stats)에 추출 통계(ext_stats)를 붙임 (Left Join)
    # 이유: 추출되지 않은 그룹(0개인 경우)도 확인하기 위해
    merged = pd.merge(org_stats, ext_stats, on=['Year', 'NODE_CLSS_02'], how='left')
    
    # NaN(매칭 안 된 곳)은 0으로 채움 (추출된 게 없다는 뜻)
    merged['Actual_Count'] = merged['Actual_Count'].fillna(0).astype(int)
    
    # 검증: 목표치(Target)와 실제치(Actual)가 같은가?
    merged['Is_Correct'] = merged['Target_Count'] == merged['Actual_Count']
    
    # -------------------------------------------------------
    # 4. 결과 리포트 출력
    # -------------------------------------------------------
    print("\n" + "="*70)
    print(f"{'Year':<6} | {'Category':<15} | {'Total':<7} | {'Target(30%)':<11} | {'Actual':<7} | {'Status'}")
    print("="*70)
    
    all_pass = True
    
    for idx, row in merged.iterrows():
        year = row['Year']
        clss = row['NODE_CLSS_02']
        total = row['Total_Count']
        target = row['Target_Count']
        actual = row['Actual_Count']
        is_correct = row['Is_Correct']
        
        if not is_correct:
            all_pass = False
            status = "❌ Mismatch"
        else:
            status = "✅ Pass"
            
        print(f"{year:<6} | {clss:<15} | {total:<7} | {target:<11} | {actual:<7} | {status}")

    print("="*70)
    
    if all_pass:
        print("\n🎉 완벽합니다! 모든 그룹에서 정확히 상위 30% 개수만큼 추출되었습니다.")
    else:
        print("\n⚠️ 일부 그룹에서 개수가 일치하지 않습니다. 위 표를 확인해주세요.")
        print("(참고: 30% 계산값이 0인 경우, 추출 파일에 데이터가 없어야 정상입니다.)")

if __name__ == "__main__":
    verify_extraction()

🔍 검증 작업을 시작합니다...

✅ 원본 데이터 로드 완료 (62199건)
✅ 추출 데이터 로드 완료 (18638건)

📊 검증 결과 집계 중...

Year   | Category        | Total   | Target(30%) | Actual  | Status
2021   | 건축공학            | 2402    | 720         | 720     | ✅ Pass
2021   | 공학 일반           | 1128    | 338         | 338     | ✅ Pass
2021   | 기계공학            | 2635    | 790         | 790     | ✅ Pass
2021   | 기타 공학           | 447     | 134         | 134     | ✅ Pass
2021   | 산업공학            | 289     | 86          | 86      | ✅ Pass
2021   | 재료·에너지공학        | 308     | 92          | 92      | ✅ Pass
2021   | 전기전자공학          | 3697    | 1109        | 1109    | ✅ Pass
2021   | 조선해양공학          | 208     | 62          | 62      | ✅ Pass
2021   | 컴퓨터학            | 1327    | 398         | 398     | ✅ Pass
2021   | 화학공학            | 333     | 99          | 99      | ✅ Pass
2022   | 건축공학            | 2333    | 699         | 699     | ✅ Pass
2022   | 공학 일반           | 1090    | 327         | 327     | ✅ Pass
2022   | 기계공학            | 2591

In [2]:
# 상위 70%로 변경

import json
import pandas as pd
import os
import re

# -----------------------------------------------------------
# 1. 파일 경로 설정
# -----------------------------------------------------------
json_file_path = '../SSU_Datathon2025_공학분야_62199.json'

file_paths_if = {
    2021: '2021_인용지수_2년분.xls',
    2022: '2022_인용지수_2년분.xls',
    2023: '2023_인용지수_2년분.xls',
    2024: '2024_인용지수_2년분.xls'
}

# -----------------------------------------------------------
# 2. 사용자 수기 확인 내역 (매핑 테이블) 적용
# -----------------------------------------------------------
manual_mapping = {
    "(사)한국CDE학회": "한국CDE학회",
    "ICT플랫폼학회": "아이씨티플랫폼학회",
    "유공압건설기계학회": "사단법인 유공압건설기계학회",
    "한국로봇학회(논문지)": "한국로봇학회",
    "한국염색가공학회": "한국염색가공학회",
    "한국위험물학회": "한국위험물학회",
    "한국자동차안전학회": "사단법인 한국자동차안전학회",
    "한국전자파학회JEES": "한국전자파학회",
    "한국정보통신학회JICCE": "한국정보통신학회",
    "한국컴퓨터그래픽스학회": "(사)한국컴퓨터그래픽스학회",
    "한국콘텐츠학회(IJOC)": "한국콘텐츠학회",
    "한국환경에너지공학회": "(사)한국환경에너지공학회"
}

# -----------------------------------------------------------
# 3. 유틸리티 함수
# -----------------------------------------------------------
def normalize_name(name):
    """매칭 확률을 높이기 위해 공백과 특수문자를 제거"""
    if pd.isna(name): return ""
    return re.sub(r'[^a-zA-Z0-9가-힣]', '', str(name).upper())

def load_if_database(file_paths):
    """인용지수 DB 구축"""
    db = {}
    print("📂 인용지수 데이터베이스 구축 중...")
    
    for year, path in file_paths.items():
        if not os.path.exists(path):
            db[year] = {}
            continue
            
        try:
            df = pd.read_excel(path, engine='xlrd')
            df.columns = df.columns.str.replace('\n', '').str.strip()
            
            # IF 컬럼 찾기
            if_cols = [c for c in df.columns if '2년' in c and 'IF' in c]
            if not if_cols:
                db[year] = {}
                continue
            if_col = if_cols[0]
            
            # 발행기관 컬럼 찾기
            org_cols = [c for c in df.columns if any(x in c for x in ['발행기관', '발행처', '학회'])]
            target_col = org_cols[0] if org_cols else (df.columns[1] if len(df.columns) > 1 else df.columns[0])

            # 점수 매핑
            mapping = {}
            df[if_col] = pd.to_numeric(df[if_col], errors='coerce').fillna(0)
            
            for idx, row in df.iterrows():
                org_name = row[target_col]
                score = row[if_col]
                norm_name = normalize_name(org_name)
                
                if norm_name:
                    mapping[norm_name] = max(mapping.get(norm_name, 0), score)
            
            db[year] = mapping
            
        except Exception as e:
            print(f"   ❌ {year}년 로드 실패: {e}")
            db[year] = {}
            
    db[2025] = db.get(2024, {})
    return db

def main():
    # 1. 인용지수 DB 로드
    if_db = load_if_database(file_paths_if)
    
    # 2. JSON 논문 데이터 로드
    print(f"📂 논문 데이터 로드 중... ({json_file_path})")
    try:
        with open(json_file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
        if "NODE_LIST" not in data:
            print("오류: NODE_LIST 키가 없습니다.")
            return

        df = pd.DataFrame(data["NODE_LIST"])
        
        # 분석을 위해 임시로 Year 컬럼 생성
        df['temp_Year'] = df['PBSH'].astype(str).str.strip().str[:4]
        
        # 3. IF 점수 매핑 (보정 로직 적용)
        print("📊 보정된 매핑 테이블을 사용하여 IF 점수 계산 중...")
        
        def get_score(row):
            try:
                y = int(row['temp_Year'])
            except:
                y = 0
            
            original_name = row['IPRD_NM']
            
            # 1. 수기 매핑 테이블 확인
            if original_name in manual_mapping:
                search_name = manual_mapping[original_name]
            else:
                search_name = original_name
                
            # 2. 정규화 및 점수 조회
            norm_name = normalize_name(search_name)
            return if_db.get(y, {}).get(norm_name, 0)

        df['IF_Score'] = df.apply(get_score, axis=1)
        
        # 4. 상위 70% 추출 (변경됨)
        print("✂️ [연도 x 중분류] 별 상위 70% 필터링 시작...")
        
        filtered_results = []
        groups = df.groupby(['temp_Year', 'NODE_CLSS_02'])
        
        for (year, clss), group in groups:
            total_count = len(group)
            
            # [변경됨] 상위 30% -> 상위 70% (0.3 -> 0.7)
            target_count = int(total_count * 0.7)
            
            if target_count == 0:
                continue

            # 정렬: IF 점수 내림차순 -> NODE_ID 오름차순
            sorted_group = group.sort_values(by=['IF_Score', 'NODE_ID'], ascending=[False, True])
            
            # 자르기
            top_70 = sorted_group.head(target_count)
            filtered_results.append(top_70)

        # 5. 저장
        if filtered_results:
            final_df = pd.concat(filtered_results)
            
            # 원본 데이터 형태 유지를 위해 임시 컬럼(temp_Year) 삭제
            if 'temp_Year' in final_df.columns:
                final_df = final_df.drop(columns=['temp_Year'])
            
            # [변경됨] 파일명 수정 (30 -> 70)
            output_json = 'top_70_percent_corrected.json'
            
            # JSON 저장
            final_df.to_json(output_json, orient='records', force_ascii=False, indent=4)
            
            print("\n" + "="*50)
            print(f"🎉 저장 완료! (수기 보정 적용됨)")
            print(f"- 원본 데이터: {len(df)}건")
            print(f"- 추출 데이터: {len(final_df)}건 (상위 70%)")
            print(f"- 파일명: {output_json}")
            print("="*50)
            
            # 저장된 데이터의 컬럼 목록 확인
            print("[저장된 필드 목록]")
            print(final_df.columns.tolist())
            
        else:
            print("조건에 맞는 논문이 없습니다.")

    except Exception as e:
        print(f"에러 발생: {e}")

if __name__ == "__main__":
    main()

📂 인용지수 데이터베이스 구축 중...
WARNING *** OLE2 stream 'SSCS': expected size 132288, actual size 512
WARNING *** OLE2 stream 'SSCS': expected size 133824, actual size 512
WARNING *** OLE2 stream 'SSCS': expected size 132288, actual size 512
WARNING *** OLE2 stream 'SSCS': expected size 133312, actual size 512
📂 논문 데이터 로드 중... (../SSU_Datathon2025_공학분야_62199.json)
📊 보정된 매핑 테이블을 사용하여 IF 점수 계산 중...
✂️ [연도 x 중분류] 별 상위 70% 필터링 시작...

🎉 저장 완료! (수기 보정 적용됨)
- 원본 데이터: 62199건
- 추출 데이터: 43514건 (상위 70%)
- 파일명: top_70_percent_corrected.json
[저장된 필드 목록]
['NODE_ID', 'IPRD_NM', 'PLCT_NM', 'NODE_TTLE', 'NODE_TTLE_EN', 'PBSH', 'NODE_LINK', 'NODE_CLSS_01', 'NODE_CLSS_02', 'AUTR_NM', 'KYWD', 'ABST_KR', 'ABST_EN', 'IF_Score']


In [4]:
import json
import pandas as pd
import math

# 1. 파일 경로 설정
original_file = '../SSU_Datathon2025_공학분야_62199.json'   # 원본 파일
extracted_file = 'top_70_percent_corrected.json'           # [수정됨] 방금 만든 70% 추출 파일

def verify_extraction():
    print("🔍 검증 작업을 시작합니다... (기준: 상위 70%)\n")

    # -------------------------------------------------------
    # 1. 원본 데이터 분석 (목표치 계산)
    # -------------------------------------------------------
    try:
        with open(original_file, 'r', encoding='utf-8') as f:
            data_org = json.load(f)
            df_org = pd.DataFrame(data_org["NODE_LIST"])
            
        # 연도 추출
        df_org['Year'] = df_org['PBSH'].astype(str).str.strip().str[:4]
        
        # 그룹별 전체 개수 카운트
        # reset_index를 해서 데이터프레임으로 만듦
        org_stats = df_org.groupby(['Year', 'NODE_CLSS_02']).size().reset_index(name='Total_Count')
        
        # [수정됨] 목표 개수(Target) 계산: 전체 * 0.7 (소수점 내림)
        org_stats['Target_Count'] = (org_stats['Total_Count'] * 0.7).astype(int)
        
        print(f"✅ 원본 데이터 로드 완료 ({len(df_org)}건)")

    except Exception as e:
        print(f"❌ 원본 파일 로드 실패: {e}")
        return

    # -------------------------------------------------------
    # 2. 추출된 데이터 분석 (실제값 확인)
    # -------------------------------------------------------
    try:
        # JSON 파일 로드
        df_ext = pd.read_json(extracted_file)
        
        # 연도 추출 (만약 저장할 때 temp_Year를 지웠다면 다시 PBSH에서 추출)
        if 'Year' not in df_ext.columns:
             df_ext['Year'] = df_ext['PBSH'].astype(str).str.strip().str[:4]

        # 그룹별 추출 개수 카운트
        ext_stats = df_ext.groupby(['Year', 'NODE_CLSS_02']).size().reset_index(name='Actual_Count')
        
        print(f"✅ 추출 데이터 로드 완료 ({len(df_ext)}건)")

    except ValueError:
        print("❌ 추출된 파일이 비어있거나 형식이 잘못되었습니다.")
        return
    except Exception as e:
        print(f"❌ 추출 파일 로드 실패: {e}")
        return

    # -------------------------------------------------------
    # 3. 비교 검증 (Merge & Compare)
    # -------------------------------------------------------
    print("\n📊 검증 결과 집계 중...")
    
    # 원본 통계(org_stats)에 추출 통계(ext_stats)를 붙임 (Left Join)
    # 이유: 추출되지 않은 그룹(0개인 경우)도 확인하기 위해
    merged = pd.merge(org_stats, ext_stats, on=['Year', 'NODE_CLSS_02'], how='left')
    
    # NaN(매칭 안 된 곳)은 0으로 채움 (추출된 게 없다는 뜻)
    merged['Actual_Count'] = merged['Actual_Count'].fillna(0).astype(int)
    
    # 검증: 목표치(Target)와 실제치(Actual)가 같은가?
    merged['Is_Correct'] = merged['Target_Count'] == merged['Actual_Count']
    
    # -------------------------------------------------------
    # 4. 결과 리포트 출력
    # -------------------------------------------------------
    print("\n" + "="*70)
    # [수정됨] 헤더 30% -> 70%
    print(f"{'Year':<6} | {'Category':<15} | {'Total':<7} | {'Target(70%)':<11} | {'Actual':<7} | {'Status'}")
    print("="*70)
    
    all_pass = True
    
    for idx, row in merged.iterrows():
        year = row['Year']
        clss = row['NODE_CLSS_02']
        total = row['Total_Count']
        target = row['Target_Count']
        actual = row['Actual_Count']
        is_correct = row['Is_Correct']
        
        if not is_correct:
            all_pass = False
            status = "❌ Mismatch"
        else:
            status = "✅ Pass"
            
        print(f"{year:<6} | {clss:<15} | {total:<7} | {target:<11} | {actual:<7} | {status}")

    sum_total = merged['Total_Count'].sum()
    sum_target = merged['Target_Count'].sum()
    sum_actual = merged['Actual_Count'].sum()

    # 구분선 (이중선 대신 단선으로 구별)
    print("-" * 70)
    
    # 총합 행 출력
    print(f"{'TOTAL':<6} | {'ALL':<15} | {sum_total:<7} | {sum_target:<11} | {sum_actual:<7} | -")
    
    print("="*70)
    
    if all_pass:
        print("\n🎉 완벽합니다! 모든 그룹에서 정확히 상위 70% 개수만큼 추출되었습니다.")
    else:
        print("\n⚠️ 일부 그룹에서 개수가 일치하지 않습니다. 위 표를 확인해주세요.")
        print("(참고: 70% 계산값이 0인 경우, 추출 파일에 데이터가 없어야 정상입니다.)")

if __name__ == "__main__":
    verify_extraction()

🔍 검증 작업을 시작합니다... (기준: 상위 70%)

✅ 원본 데이터 로드 완료 (62199건)
✅ 추출 데이터 로드 완료 (43514건)

📊 검증 결과 집계 중...

Year   | Category        | Total   | Target(70%) | Actual  | Status
2021   | 건축공학            | 2402    | 1681        | 1681    | ✅ Pass
2021   | 공학 일반           | 1128    | 789         | 789     | ✅ Pass
2021   | 기계공학            | 2635    | 1844        | 1844    | ✅ Pass
2021   | 기타 공학           | 447     | 312         | 312     | ✅ Pass
2021   | 산업공학            | 289     | 202         | 202     | ✅ Pass
2021   | 재료·에너지공학        | 308     | 215         | 215     | ✅ Pass
2021   | 전기전자공학          | 3697    | 2587        | 2587    | ✅ Pass
2021   | 조선해양공학          | 208     | 145         | 145     | ✅ Pass
2021   | 컴퓨터학            | 1327    | 928         | 928     | ✅ Pass
2021   | 화학공학            | 333     | 233         | 233     | ✅ Pass
2022   | 건축공학            | 2333    | 1633        | 1633    | ✅ Pass
2022   | 공학 일반           | 1090    | 763         | 763     | ✅ Pass
2022   | 기계공학     